# Foldseek Toolkit

This notebook demonstrates the five Foldseek tools:

1. **`foldseek-search`** — single-chain structural homology search (remote default)
2. **`foldseek-cluster`** — cluster a set of structures by structural similarity (local-only)
3. **`foldseek-multimer-search`** — multimer (complex) structural search (remote default)
4. **`foldseek-multimercluster`** — cluster a set of multi-chain assemblies by complex-level structural similarity (local-only)
5. **`foldseek-rbh`** — reciprocal-best-hits structural search between a query and a target DB (local-only)

References:
- van Kempen et al., *Nature Biotechnology* 2024 ([DOI](https://doi.org/10.1038/s41587-023-01773-0)) — Foldseek
- Kim et al., *Nature Methods* 2025 ([DOI](https://doi.org/10.1038/s41592-025-02593-7)) — Foldseek-Multimer

In [1]:
from proto_tools.tools.database_retrieval import (
    AlphaFoldDBFetchConfig, AlphaFoldDBFetchInput, run_alphafold_db_fetch,
)
from proto_tools.tools.structure_alignment import (
    FoldseekClusterConfig, FoldseekClusterInput, run_foldseek_cluster,
    FoldseekMultimerClusterConfig, FoldseekMultimerClusterInput, run_foldseek_multimercluster,
    FoldseekMultimerSearchConfig, FoldseekMultimerSearchInput, run_foldseek_multimer_search,
    FoldseekRBHConfig, FoldseekRBHInput, run_foldseek_rbh,
    FoldseekSearchConfig, FoldseekSearchInput, run_foldseek_search,
)

## Input API

| Field | Type | Description |
|-------|------|-------------|
| `structure_text` | `str` | PDB-format text of the query structure. |

## Config API

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `databases` | `list[str]` | `["pdb100", "afdb50"]` | Foldseek reference databases. |
| `mode` | `Literal["3diaa", "tmalign", "lolalign"]` | `"3diaa"` | Alignment mode. |
| `poll_interval_seconds` | `float` | `5.0` | Delay between status polls. |
| `timeout_seconds` | `float` | `600.0` | Max wall-clock time for the search. |

## Output API

**`FoldseekSearchOutput`:**

| Field | Type | Description |
|-------|------|-------------|
| `ticket_id` | `str` | Foldseek job ticket ID. |
| `hits` | `list[FoldseekHit]` | All alignment hits across queried databases. |
| `num_hits` | `int` | `len(hits)`. |
| `databases_queried` | `list[str]` | Databases included in the search. |
| `result_url` | `str` | Foldseek result-archive download URL. |

**`FoldseekHit`** has: `database`, `target_id`, `sequence_identity`, `alignment_length`, `mismatches`, `gap_openings`, `query_start`, `query_end`, `target_start`, `target_end`, `evalue`, `bit_score`.

## Step 1: fetch a query structure (TP53 from AFDB)

In [2]:
afdb = run_alphafold_db_fetch(
    AlphaFoldDBFetchInput(uniprot_id="P04637"),
    AlphaFoldDBFetchConfig(structure_format="pdb"),
)
print(f"Fetched {afdb.entry_id}: {afdb.sequence_length} residues, mean pLDDT={afdb.mean_plddt:.1f}")

Fetched AF-P04637-F1: 393 residues, mean pLDDT=75.1


## Step 2: search Foldseek (PDB100 only, ~30-60s)

In [3]:
output = run_foldseek_search(
    FoldseekSearchInput(structure_text=afdb.structure.structure_pdb),
    FoldseekSearchConfig(databases=["pdb100"]),
)
print(f"Foldseek returned {output.num_hits} hits across {output.databases_queried}")

Foldseek returned 890 hits across ['pdb100']


## Step 3: inspect top hits ranked by E-value

In [4]:
top_hits = sorted(output.hits, key=lambda h: h.evalue)[:10]
for hit in top_hits:
    print(
        f"{hit.target_id:<20} db={hit.database:<8} "
        f"evalue={hit.evalue:.2e}  identity={hit.sequence_identity:.1%}  "
        f"aligned={hit.alignment_length} aa"
    )

7t34-assembly1.cif.gz_A Crystal structure of AtDHDPR1 from Arabidopsis thaliana db=pdb100   evalue=2.90e-02  identity=8.5%  aligned=47 aa
4gkp-assembly1.cif.gz_A Structure of the truncated neck and C-terminal motor homology domain of ViK1 from Candida glabrata db=pdb100   evalue=3.40e-02  identity=4.9%  aligned=142 aa
5u5n-assembly1.cif.gz_B The dimeric crystal structure of HTPA Reductase from Sellaginella moellendorffii db=pdb100   evalue=3.40e-02  identity=8.1%  aligned=49 aa
3m7o-assembly3.cif.gz_C Crystal structure of mouse MD-1 in complex with phosphatidylcholine db=pdb100   evalue=4.50e-02  identity=9.1%  aligned=197 aa
5lgv-assembly1.cif.gz_B GlgE isoform 1 from Streptomyces coelicolor E423A mutant soaked in maltooctaose db=pdb100   evalue=4.50e-02  identity=6.3%  aligned=126 aa
5vt4-assembly2.cif.gz_C Sco GlgEI-V279S in complex with a pyrolidene-based methyl-phosphonate compound db=pdb100   evalue=5.10e-02  identity=6.3%  aligned=126 aa
4cn1-assembly1.cif.gz_B GlgE isoform 1 fr

## Cluster a candidate set of structures

`foldseek-cluster` groups a list of PDB structures by structural similarity. Local-only — runs the `foldseek easy-cluster` binary in a standalone subprocess.

In [5]:
cluster_output = run_foldseek_cluster(
    FoldseekClusterInput(
        structures=[afdb.structure.structure_pdb, afdb.structure.structure_pdb],
        structure_ids=["tp53_copy1", "tp53_copy2"],
    ),
    FoldseekClusterConfig(),
)
print(f"{cluster_output.num_clusters} cluster(s) over {cluster_output.num_structures} structure(s):")
for cluster in cluster_output.clusters:
    print(f"  {cluster.representative_id}: {cluster.member_ids}")

Running run_foldseek_cluster [00:00]

1 cluster(s) over 2 structure(s):
  tp53_copy2: ['tp53_copy2', 'tp53_copy1']


## Multimer (complex) search

`foldseek-multimer-search` takes a multi-chain PDB and searches against multimer-aware databases. Remote mode wraps the alignment mode as `complex-{base}` per the Foldseek server's wire protocol.

In [6]:
import requests

# Fetch a real multi-chain complex from RCSB: hemoglobin (4HHB), an alpha2-beta2 tetramer.
multimer_pdb = requests.get("https://files.rcsb.org/download/4HHB.pdb", timeout=30).text

multi_output = run_foldseek_multimer_search(
    FoldseekMultimerSearchInput(structure_text=multimer_pdb),
    FoldseekMultimerSearchConfig(databases=["pdb100"]),
)
print(f"Multimer search returned {multi_output.num_hits} hits")
for hit in sorted(multi_output.hits, key=lambda h: h.evalue)[:5]:
    print(f"  {hit.target_id}: evalue={hit.evalue:.2e}, identity={hit.sequence_identity:.1%}")

Multimer search returned 1912 hits
  1a00-assembly1.cif.gz_A HEMOGLOBIN (VAL BETA1 MET, TRP BETA37 TYR) MUTANT: evalue=1.00e+00, identity=100.0%
  1a01-assembly1.cif.gz_A HEMOGLOBIN (VAL BETA1 MET, TRP BETA37 ALA) MUTANT: evalue=1.00e+00, identity=100.0%
  1a0u-assembly1.cif.gz_C HEMOGLOBIN (VAL BETA1 MET) MUTANT: evalue=1.00e+00, identity=100.0%
  1a0z-assembly1.cif.gz_A HEMOGLOBIN (VAL BETA1 MET) MUTANT: evalue=1.00e+00, identity=100.0%
  3a0g-assembly1.cif.gz_A Crystal structure analysis of guinea pig oxyhemoglobin at 2.5 angstroms resolution: evalue=1.00e+00, identity=75.5%


## Multimer clustering

`foldseek-multimercluster` groups a set of multi-chain assemblies by combined chain + interface similarity. Local-only — clustering an arbitrary user set of complexes is not exposed by the public server.

In [7]:
# Cluster three real multimers from RCSB: PD-1/PD-L1 (3BIK, hetero-trimer)
# and two TIM dimers (1TIM, 8TIM). Expect two clusters: 3BIK alone, the two
# structurally homologous TIMs together.
multimer_pdbs = [
    requests.get(f"https://files.rcsb.org/download/{pdb}.pdb", timeout=30).text
    for pdb in ("3BIK", "1TIM", "8TIM")
]

mc_output = run_foldseek_multimercluster(
    FoldseekMultimerClusterInput(structures=multimer_pdbs, structure_ids=["3BIK", "1TIM", "8TIM"]),
    FoldseekMultimerClusterConfig(num_threads=2),
)
print(f"{mc_output.num_clusters} multimer clusters across {mc_output.num_multimers} input complexes")
for cluster in mc_output.clusters:
    print(f"  representative {cluster.representative_id}: {len(cluster.member_ids)} members ({cluster.member_ids})")

Running run_foldseek_multimercluster [00:00]

2 multimer clusters across 3 input complexes
  representative 3BIK: 1 members (['3BIK'])
  representative 8TIM: 2 members (['8TIM', '1TIM'])


## Reciprocal best hits

`foldseek-rbh` returns only mutual best hits between a query and a target DB — useful for structural orthology calls. Local-only (the public server does not expose RBH). Pass either a prebuilt Foldseek DB or a directory of PDBs (Foldseek auto-builds a temporary DB).

In [8]:
# Build a tiny target set from real PDBs and run RBH against it.
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    target_dir = Path(tmp)
    for pdb_id in ("1TIM", "8TIM"):
        pdb_text = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=30).text
        (target_dir / f"{pdb_id}.pdb").write_text(pdb_text)

    rbh_query = requests.get("https://files.rcsb.org/download/1TIM.pdb", timeout=30).text

    rbh_output = run_foldseek_rbh(
        FoldseekRBHInput(structure_text=rbh_query),
        FoldseekRBHConfig(local_db=str(target_dir), num_threads=2),
    )

print(f"RBH returned {rbh_output.num_hits} mutual best hits")
for hit in rbh_output.hits:
    print(f"  {hit.target_id}: evalue={hit.evalue:.2e}, identity={hit.sequence_identity:.1%}")

Running run_foldseek_rbh [00:00]

RBH returned 2 mutual best hits
  1TIM_A: evalue=2.37e-55, identity=100.0%
  1TIM_B: evalue=1.32e-54, identity=100.0%
